# NER: Location and Date Extraction from German News Texts

This notebook extracts two kinds of information from OCR'd German news texts:

- **Where**: geographic locations (places of origin/dispatch), using the `deepset/bert-base-german-cased-ner` model
- **When**: dates, using regex patterns matched against German newspaper dateline conventions

It is designed to run as a Galaxy JupyTool step following Tesseract ATR output, but can also be run standalone.

## Inputs (Galaxy)
- `textDataset`: path to a `.txt` file (Tesseract output)

## Outputs (Galaxy)
- `output_dataset`: a JSONL file where each line is a JSON object `{"source": <filename>, "locations": [...], "dates": [...], "datelines": [...]}`

In [ ]:
import os
import json
import re
import sys
from pathlib import Path

## 1. Environment detection

Detect whether we are running inside Galaxy (JupyTool) or standalone (local testing).

In [ ]:
# Detect Galaxy environment
GALAXY_INPUTS_RAW = os.environ.get("GALAXY_INPUTS", None)
IN_GALAXY = GALAXY_INPUTS_RAW is not None

print(f"Running in Galaxy: {IN_GALAXY}")

if IN_GALAXY:
    GALAXY_INPUTS = json.loads(GALAXY_INPUTS_RAW)
    print("Galaxy inputs:", json.dumps(GALAXY_INPUTS, indent=2))
else:
    print("Standalone mode: using TEST_INPUT_PATH variable below.")

## 2. Input configuration

In Galaxy the input text path is read from `GALAXY_INPUTS`. In standalone mode, set `TEST_INPUT_PATH` to a local `.txt` file for testing.

In [ ]:
# --- Standalone testing: set this path to a local .txt file ---
TEST_INPUT_PATH = "sample_article.txt"

# Resolve the actual input path
if IN_GALAXY:
    # Galaxy passes a dataset as a list of {path, name} dicts
    input_path = GALAXY_INPUTS["textDataset"][0]["path"]
    input_name = GALAXY_INPUTS["textDataset"][0].get("name", Path(input_path).name)
else:
    input_path = TEST_INPUT_PATH
    input_name = Path(input_path).name

print(f"Input file : {input_path}")
print(f"Input label: {input_name}")

## 3. Read input text

In [ ]:
with open(input_path, "r", encoding="utf-8", errors="replace") as f:
    full_text = f.read()

print(f"Read {len(full_text)} characters.")
print("--- Preview (first 500 chars) ---")
print(full_text[:500])

## 4. Date extraction (regex)

German newspapers, especially older ones, use consistent dateline formats. We match these with regex rather than ML to keep things fast and transparent.

Patterns covered:
- Full datelines: `Wien, 12. März 1920.` / `BERLIN, den 5. April 1933`
- Bare dates: `12. März 1920`, `15. Jänner 1848`
- Year-only mentions like `im Jahre 1920`

`Jänner` (Austrian/South German for January) is included alongside `Januar`.

In [ ]:
GERMAN_MONTHS = (
    r"Jänner|Januar|Februar|März|April|Mai|Juni|"
    r"Juli|August|September|Oktober|November|Dezember"
)

# Pattern 1: full dateline  "Wien, 12. März 1920" or "WIEN, den 5. April 1933"
DATELINE_RE = re.compile(
    r"([A-ZÄÖÜ][A-Za-zäöüÄÖÜß]+(?:[\s\-][A-Za-zäöüÄÖÜß]+)*),\s+"
    r"(?:den\s+)?(\d{1,2}\.\s+(?:" + GERMAN_MONTHS + r")(?:\.?\s+\d{2,4})?)",
    re.UNICODE
)

# Pattern 2: bare date  "12. März 1920"  or  "5. Jänner"
DATE_RE = re.compile(
    r"\b(\d{1,2}\.\s*(?:" + GERMAN_MONTHS + r")(?:\.?\s+\d{2,4})?)",
    re.UNICODE
)

# Extract datelines (city + date together)
datelines = []
for m in DATELINE_RE.finditer(full_text):
    datelines.append({"city": m.group(1).strip(), "date": m.group(2).strip(), "raw": m.group(0)})

# Extract all date mentions
dates = [m.group(0).strip() for m in DATE_RE.finditer(full_text)]
# Deduplicate while preserving order
seen = set()
unique_dates = [d for d in dates if not (d in seen or seen.add(d))]

print(f"Datelines found : {len(datelines)}")
for dl in datelines:
    print(f"  {dl['raw']}")

print(f"\nDate mentions found: {len(unique_dates)}")
for d in unique_dates[:10]:
    print(f"  {d}")

## 5. Location NER (transformers)

We use `deepset/bert-base-german-cased-ner`, which was fine-tuned on German CoNLL-2003 data and recognises `LOC`, `PER`, `ORG`, and `OTH` entity types. We keep only `LOC` spans here.

The model is downloaded from HuggingFace Hub on first run and cached. In a persistent Galaxy JupyTool environment the cache survives between runs.

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

MODEL_NAME = "deepset/bert-base-german-cased-ner"

print(f"Loading model: {MODEL_NAME} ...")
ner_pipeline = pipeline(
    "ner",
    model=MODEL_NAME,
    aggregation_strategy="simple",   # merges B-/I- tokens into full spans
    device=-1                        # CPU; change to 0 for GPU if available
)
print("Model loaded.")

In [ ]:
# BERT has a 512-token limit, so we chunk long texts on sentence boundaries.
# A safe character-level chunk size (1 token ≈ 4–5 chars for German).
CHUNK_SIZE = 1800  # characters
OVERLAP    = 200   # characters of overlap to avoid missing entities at boundaries

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Split text into overlapping chunks, preferring sentence boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        # Try to break at a sentence boundary (. ! ?)
        if end < len(text):
            last_break = max(
                text.rfind(".", start, end),
                text.rfind("!", start, end),
                text.rfind("?", start, end),
            )
            if last_break > start + chunk_size // 2:
                end = last_break + 1
        chunks.append(text[start:end])
        start = end - overlap
    return chunks


chunks = chunk_text(full_text)
print(f"Text split into {len(chunks)} chunk(s) for NER.")

In [ ]:
raw_entities = []
for i, chunk in enumerate(chunks):
    results = ner_pipeline(chunk)
    raw_entities.extend(results)
    print(f"  Chunk {i+1}/{len(chunks)}: {len(results)} entity span(s) found")

# Keep only LOC entities, deduplicate by word surface form
loc_seen = set()
locations = []
for ent in raw_entities:
    if ent["entity_group"] == "LOC":
        word = ent["word"].strip()
        if word not in loc_seen and len(word) > 1:
            loc_seen.add(word)
            locations.append({
                "text": word,
                "score": round(float(ent["score"]), 4)
            })

# Sort by confidence score descending
locations.sort(key=lambda x: x["score"], reverse=True)

print(f"\nUnique locations found: {len(locations)}")
for loc in locations[:20]:
    print(f"  {loc['text']:<30} (score: {loc['score']})")

## 6. Assemble and write output

Output is a single JSON object (written as one JSONL line) containing:
- `source`: the input filename
- `datelines`: list of `{city, date, raw}` objects extracted from dateline patterns
- `dates`: deduplicated list of all date strings found
- `locations`: deduplicated list of `{text, score}` objects for LOC entities

In [ ]:
result = {
    "source":    input_name,
    "datelines": datelines,
    "dates":     unique_dates,
    "locations": locations,
}

# Determine output path
if IN_GALAXY:
    output_path = os.environ.get("GALAXY_OUTPUTS_DIR", "/outputs") + "/output_dataset"
else:
    output_path = "ner_output.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(json.dumps(result, ensure_ascii=False) + "\n")

print(f"Output written to: {output_path}")
print(json.dumps(result, indent=2, ensure_ascii=False))

## Summary

In [ ]:
print("=== NER Summary ===")
print(f"  Source file  : {input_name}")
print(f"  Datelines    : {len(datelines)}")
print(f"  Date mentions: {len(unique_dates)}")
print(f"  Locations    : {len(locations)}")

if datelines:
    print("\n  Top dateline:")
    print(f"    {datelines[0]['city']} — {datelines[0]['date']}")

if locations:
    print("\n  Top location(s):")
    for loc in locations[:5]:
        print(f"    {loc['text']}")